In [ ]:
import pandas as pd
from collections import defaultdict

# ============================================================
# Compute Elo ratings for every international team over history
# ============================================================

# 1. Load and prepare
df = df_games.copy()
df = df.dropna(subset=['home_score', 'away_score'])     # drop unplayed 2026 fixtures
df = df.sort_values('date').reset_index(drop=True)      # chronological order is essential

# 2. K-factor by tournament importance (controls how reactive Elo is)
def get_k_factor(tournament):
    if tournament == 'FIFA World Cup':
        return 60
    if 'qualification' in tournament.lower() or tournament == 'UEFA Nations League':
        return 40
    if tournament == 'Friendly':
        return 20
    # Continental cups (Euro, Copa America, AFCON, Asian Cup, etc.)
    return 50

# 3. Expected score formula — probability home team wins given Elo difference
def expected_score(home_elo, away_elo, home_advantage=100):
    # home_advantage = 100 is the standard adjustment in football Elo
    # (~64% win rate for evenly matched teams playing at home)
    diff = (home_elo + home_advantage) - away_elo
    return 1 / (1 + 10 ** (-diff / 400))

# 4. Initialize and iterate
INITIAL_ELO = 1500
current_elo = defaultdict(lambda: INITIAL_ELO)
elo_history = []   # will store (date, team, elo_after_match) rows

for row in df.itertuples(index=False):
    home, away = row.home_team, row.away_team
    home_elo, away_elo = current_elo[home], current_elo[away]

    # Neutral venue cancels home advantage
    h_adv = 0 if row.neutral else 100
    exp_home = expected_score(home_elo, away_elo, h_adv)
    exp_away = 1 - exp_home

    # Actual result
    if row.home_score > row.away_score:
        actual_home, actual_away = 1.0, 0.0
    elif row.home_score < row.away_score:
        actual_home, actual_away = 0.0, 1.0
    else:
        actual_home, actual_away = 0.5, 0.5

    # Goal-difference multiplier — bigger wins move Elo more (standard football Elo)
    goal_diff = abs(row.home_score - row.away_score)
    if goal_diff <= 1:
        g = 1.0
    elif goal_diff == 2:
        g = 1.5
    else:
        g = (11 + goal_diff) / 8

    K = get_k_factor(row.tournament)

    # Update
    current_elo[home] = home_elo + K * g * (actual_home - exp_home)
    current_elo[away] = away_elo + K * g * (actual_away - exp_away)

    # Log post-match Elo for both teams
    elo_history.append((row.date, home, current_elo[home]))
    elo_history.append((row.date, away, current_elo[away]))

# 5. Build the lookup table
df_elo = pd.DataFrame(elo_history, columns=['date', 'team', 'elo_after'])
df_elo['date'] = pd.to_datetime(df_elo['date'])

print(df_elo.shape)
print(df_elo.tail(10))

# 6. Sanity check — current top 15
current_ranking = (
    df_elo.sort_values('date')
          .groupby('team')
          .tail(1)
          .sort_values('elo_after', ascending=False)
          .head(15)
)
print(current_ranking)